In [ ]:
CREATE OR ALTER VIEW dbo.vw__dim__MatGrp__dedup
AS
SELECT
    m.MaterialGroupCode,
    MAX(m.MaterialGroupDescription)   AS MaterialGroupDescription,
    MAX(m.ProductDivisionCode)        AS ProductDivisionCode,
    MAX(m.ProductDivisionDescription) AS ProductDivisionDescription
FROM dbo.MaterialMaster m
WHERE m.ProductDivisionCode IS NOT NULL
  AND LTRIM(RTRIM(m.ProductDivisionCode)) <> ''
GROUP BY m.MaterialGroupCode;


In [ ]:
CREATE OR ALTER VIEW dbo.vw__CM__STIP_summary_current
AS
SELECT
    CASE
        WHEN p.CurrManufacturerCd IN ('MMC','APL','PVL','NAVM','CRI','COLABEL') THEN 'Medline Brand'
        ELSE 'Non-Medline'
    END AS Manufacturer,

    p.FiscalYearPeriodDate,
    p.SalesTypeCd,
    p.CurrProdDivCd,
    p.CurrSoldToSalesOfficeCd,
    p.CurrMatGrpCd,

    SUM(COALESCE(p.GrossSalesAmount, 0)) AS GrossSales,
    SUM(COALESCE(p.CostOfGoodsSoldAmount, 0)) AS CostOfGoodsSold,
    -SUM(COALESCE(p.GroupRebateAccrualAmount, 0)) AS GroupRebateAccrualAmount,
    -SUM(COALESCE(p.CorporateRebateAccrualAmount, 0)) AS CorporateRebateAccrualAmount,
    -SUM(COALESCE(p.CustomerIncentiveRebateAmt, 0)) AS CustomerIncentiveRebateAmt,

    m.MaterialGroupDescription,
    m.ProductDivisionCode,
    m.ProductDivisionDescription
FROM dbo.SoldToInvcProfitabilityModel p
INNER JOIN dbo.vw__dim__MatGrp__dedup m
    ON m.MaterialGroupCode = p.CurrMatGrpCd
WHERE p.FiscalYearPeriodDate IN ('2025007','2025008','2025009')
  AND p.CurrProdDivCd NOT IN ('01','11','12','13','83','90','91','92','93','94','95','96','99')
  AND p.CurrSoldToSalesOfficeCd = 'GL'
  AND (
        (p.CurrDistributorPrefixCode <> 'CM' AND p.SalesTypeCd = 'DS')
        OR p.SalesTypeCd = 'DRC'
      )
GROUP BY
    CASE
        WHEN p.CurrManufacturerCd IN ('MMC','APL','PVL','NAVM','CRI','COLABEL') THEN 'Medline Brand'
        ELSE 'Non-Medline'
    END,
    p.FiscalYearPeriodDate,
    p.SalesTypeCd,
    p.CurrProdDivCd,
    p.CurrSoldToSalesOfficeCd,
    p.CurrMatGrpCd,
    m.MaterialGroupDescription,
    m.ProductDivisionCode,
    m.ProductDivisionDescription;


In [ ]:
SELECT TOP (10) *
FROM dbo.vw__CM__STIP_summary_current
ORDER BY FiscalYearPeriodDate DESC;
